# 01. Data Preprocessing, Target Definition & Relational Merging

## Objective
The primary goal of this notebook is to transform raw, disconnected relational CSV tables from the Olist E-Commerce dataset into a clean, unified **Master Dataset** at the `order_id` level.

### Key Milestones:
1. Parse string timestamps into proper datetime objects.
2. Define our binary target variable (`is_delayed`) while avoiding data leakage.
3. Filter out invalid/canceled orders that lack delivery signals.
4. Resolve 1-to-Many relational row multiplication during dataset joins.
5. Merge geographic and product dimensions into a single master dataframe.

In [2]:
import pandas as pd
import numpy as np

# Load the core orders table
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')

# View raw schema and initial preview
print(f"Dataset shape: {orders.shape}")
print("\nData Types:")
print(orders.dtypes)
orders.head()

Dataset shape: (99441, 8)

Data Types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## Step 1: Datetime Conversion & Target Variable Definition

### Why convert to Datetime?
By default, Pandas loads dates as string (`object`) types. Strings cannot accurately measure temporal differences (e.g., `"2018-05-01"` vs `"2018-04-28"`) across months or leap years. Converting these columns to `pd.datetime64` enables accurate date arithmetic.

### Defining $y$ (`is_delayed`)
Our target variable is defined as:
$$y = \begin{cases} 1 & \text{if } \text{order\_delivered\_customer\_date} > \text{order\_estimated\_delivery\_date} \\ 0 & \text{otherwise} \end{cases}$$

### Filtering Out Canceled/Non-Delivered Orders
Orders with statuses like `canceled`, `unavailable`, or `shipped` have missing (`NaN`) delivery dates. A canceled order is an operational exception, not a late delivery. Including them introduces noise and distorts performance metrics.

In [3]:
# 1. Convert string dates to datetime objects
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# 2. Create the binary target variable
orders['is_delayed'] = (
    orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']
).astype(int)

# 3. Filter for completed, valid deliveries only
orders_clean = orders[
    (orders['order_status'] == 'delivered') & 
    (orders['order_delivered_customer_date'].notna())
].copy()

# Inspect class imbalance
print(f"Original record count: {len(orders)}")
print(f"Cleaned record count:  {len(orders_clean)}")
print("\nTarget Variable Class Distribution (%):")
print(orders_clean['is_delayed'].value_counts(normalize=True) * 100)

Original record count: 99441
Cleaned record count:  96470

Target Variable Class Distribution (%):
is_delayed
0    91.887633
1     8.112367
Name: proportion, dtype: float64


## Step 2: Preventing "Row Multiplication" during Merges

### The 1-to-Many Relational Problem
* `orders` contains **1 row per order**.
* `order_items` contains **1 row per item** in an order (e.g., an order with 3 items has 3 rows).

If we perform a standard `LEFT JOIN` on `order_id`, an order with 3 items will duplicate into 3 identical rows in our master table. This distorts metric evaluations and violates the assumption of independent training samples.

### The Solution: Pre-Aggregation
We aggregate `order_items` to the `order_id` level *before* joining:
* **Sum** `price` and `freight_value` to capture total financial cost.
* **Count** `order_item_id` to get total item quantity.
* **Select** the primary `seller_id` handling the order.

In [4]:
# Load order items table
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

# Pre-aggregate items to 1 row per order_id
items_agg = items.groupby('order_id').agg(
    total_freight_value=('freight_value', 'sum'),
    total_item_price=('price', 'sum'),
    num_items=('order_item_id', 'count'),
    seller_id=('seller_id', 'first')  # Primary seller handling the order
).reset_index()

# Merge aggregated items with cleaned orders
df_master = pd.merge(orders_clean, items_agg, on='order_id', how='inner')

print(f"Master shape after merging items: {df_master.shape}")
df_master.head(3)

Master shape after merging items: (96470, 13)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delayed,total_freight_value,total_item_price,num_items,seller_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.72,29.99,1,3504c0cb71d7fa48d967e0e4c94d59d9
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,22.76,118.70,1,289cdb325fb7e7f891c38608bf9e0962
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,19.22,159.90,1,4869f7a5dfa277a7dca6462dcf3b52b2


## Step 3: Merging Geographic & Physical Product Features

To predict shipping delays accurately, the model needs signals for:
1. **Spatial Complexity:** Customer zip code prefix vs. Seller zip code prefix.
2. **Volumetric & Weight Resistance:** Heavier, larger packages take longer to handle and ship.

We join the customer, seller, and product dimension tables to complete the master dataframe structure.

In [5]:
# Load relational metadata tables
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')

# 1. Merge Customer Zip Code Location Data
df_master = pd.merge(
    df_master, 
    customers[['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']], 
    on='customer_id', 
    how='left'
)

# 2. Merge Seller Zip Code Location Data
df_master = pd.merge(
    df_master, 
    sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']], 
    on='seller_id', 
    how='left'
)

# 3. Aggregate product physical traits per order
items_with_specs = pd.merge(items[['order_id', 'product_id']], products, on='product_id', how='left')
product_agg = items_with_specs.groupby('order_id').agg(
    product_weight_g=('product_weight_g', 'mean'),
    product_length_cm=('product_length_cm', 'mean'),
    product_height_cm=('product_height_cm', 'mean'),
    product_width_cm=('product_width_cm', 'mean')
).reset_index()

# Merge aggregated product specs into master
df_master = pd.merge(df_master, product_agg, on='order_id', how='left')

print("Final Master Dataframe Shape:", df_master.shape)
print("\nMaster Columns Overview:")
print(df_master.dtypes)

Final Master Dataframe Shape: (96470, 23)

Master Columns Overview:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
is_delayed                                int64
total_freight_value                     float64
total_item_price                        float64
num_items                                 int64
seller_id                                   str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
seller_zip_code_prefix                    int64
seller_city                                 str
seller_state                                str
product_weight_g    

In [6]:
import os

# Create interim folder if it doesn't exist
os.makedirs('../data/interim', exist_ok=True)

# Export intermediate master dataframe
df_master.to_csv('../data/interim/df_master_merged.csv', index=False)
print("Notebook 01 complete! Saved merged dataset to data/interim/df_master_merged.csv")

Notebook 01 complete! Saved merged dataset to data/interim/df_master_merged.csv
